# RDFLib Parser

`RDFlibConnection` loads a CIM RDF/XML file into an in-memory [rdflib](https://rdflib.readthedocs.io/) graph and runs the same SPARQL queries as the Blazegraph and GraphDB connections (it is a `SPARQLEndpointConnection`). No database server is needed, so it is handy for testing SPARQL-based code against a local file.

## Installation

The connection needs the `rdflib` extra, which installs `rdflib` and the `oxrdflib` Oxigraph store:

```bash
pip install "cim-graph[rdflib]"
```

Without it, `from cimgraph.databases import RDFlibConnection` raises an `ImportError` naming the extra to install.

## Loading a model

Select the CIM profile, then pass the path of a local RDF/XML file. The connection parses the file when it is created.

**Parameters**

* `filename` (str): a local CIM RDF/XML file. A path that is not a local file raises `FileNotFoundError`.
* `use_oxigraph` (bool, default `True`): use the Oxigraph store from `oxrdflib`; `False` uses rdflib's default in-memory store.

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim

from cimgraph.databases import RDFlibConnection

database = RDFlibConnection(filename='../../tests/test_models/ieee13.xml')
print(f'{len(database.connection_obj)} triples loaded')

3392 triples loaded


The connection is used like any other: pass it to a graph model with the feeder to load.

In [2]:
from cimgraph.models import FeederModel

feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(container=feeder, connection=database, distributed=False)

for cim_class, objects in network.graph.items():
    print(f'{cim_class.__name__}: {len(objects)}')

Feeder: 1
TransformerTank: 4
EnergyConsumer: 16
ConnectivityNode: 22
Terminal: 70
ACLineSegment: 11
PowerTransformer: 4
PowerElectronicsConnection: 5
LoadBreakSwitch: 2
Recloser: 1
LinearShuntCompensator: 2
Breaker: 1
EnergySource: 1
Fuse: 1


## Expanding the graph

`get_all_edges` queries the attributes and associations of every object of a class. Unit values are `CIMUnit`s in base SI units; convert them with `.to()`.

In [3]:
network.get_all_edges(cim.ACLineSegment)

for line in network.list_by_class(cim.ACLineSegment)[:3]:
    print(f'{line.name}: {line.length.to("ft"):.0f} ft')

645646: 300 ft
632645: 500 ft
670671: 1333 ft


## Running SPARQL queries

`execute` runs a SPARQL query on the in-memory graph. The result has the SPARQL JSON layout `{'results': {'bindings': [...]}}`, where each binding is an rdflib result row.

In [4]:
query = """
PREFIX cim: <http://iec.ch/TC57/CIM100#>
SELECT ?name WHERE {
    ?switch a cim:LoadBreakSwitch ;
            cim:IdentifiedObject.name ?name .
}
ORDER BY ?name
"""
result = database.execute(query)
print([str(row.name) for row in result['results']['bindings']])

['671692', 'sect1']


## Limitations

* Only local RDF/XML files are read, not JSON-LD or Turtle, and URLs are not fetched.
* Files with `rdf:about="urn:uuid:..."` identifiers (`CIMG_IEC61970_552='552-NEW'`, the default) work. For `552-LEGACY` and `CGMES` files (`rdf:ID="_..."`) the queries build object IRIs from `CIMG_URL`, while rdflib resolves `rdf:ID` against the file's location, so objects other than the container are not found.
* `get_all_edges` runs its batch queries one after another, not in parallel as on Blazegraph and GraphDB.

See the [Databases Overview](3_1_databases_overview.ipynb) for the methods shared by all SPARQL connections.